In [ ]:
!pip install diffusers transformers accelerate -q
!pip install opencv-python-headless -q

import torch
import numpy as np
import pandas as pd
import json
import os
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

BASE         = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER = f'{BASE}/images_human_fixed'
GEN_FOLDER   = f'{BASE}/images_generated'
RESULT_FOLDER= f'{BASE}/results'

os.makedirs(GEN_FOLDER, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
import torch


pipe = StableDiffusionInpaintPipeline.from_pretrained(
    'runwayml/stable-diffusion-inpainting',
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()

In [ ]:
train_df = pd.read_csv(
    f'{BASE}/labels/human_train.csv'
)

perfect_df = train_df[
    (train_df['domain_A'] == 2) &
    (train_df['domain_B'] == 2) &
    (train_df['domain_C'] == 2) &
    (train_df['domain_D'] == 2) &
    (train_df['domain_E'] == 2)
]

print(f'รูป perfect (all class 2): {len(perfect_df)} รูป')

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
samples = perfect_df.sample(
    min(5, len(perfect_df)), random_state=42
)

for idx, (_, row) in enumerate(samples.iterrows()):
    src = f'{HUMAN_FOLDER}/{row["filename"]}'
    try:
        img = Image.open(src).convert('RGB')
        axes[idx].imshow(img)
        axes[idx].set_title(
            f'{row["filename"][:12]}\n'
            f'A={row["domain_A"]} B={row["domain_B"]}\n'
            f'C={row["domain_C"]} D={row["domain_D"]} '
            f'E={row["domain_E"]}',
            fontsize=7
        )
    except:
        axes[idx].text(0.5, 0.5, 'Error',
                      ha='center', va='center')
    axes[idx].axis('off')

plt.suptitle('Base Images (All Class 2)',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import cv2

def create_hands_mask(img_pil, dilation=30):
    w, h  = img_pil.size
    mask  = Image.new('L', (w, h), 0)
    draw  = ImageDraw.Draw(mask)

    cx, cy = w//2, h//2
    r      = min(w, h) // 4

    draw.ellipse(
        [cx-r, cy-r, cx+r, cy+r],
        fill=255
    )
    return mask

def create_numbers_mask(img_pil, positions='random'):
    w, h = img_pil.size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)

    cx, cy = w//2, h//2
    r      = min(w, h) // 3

    angles = [
        -90, -60, -30, 0, 30, 60,
         90, 120, 150, 180, 210, 240
    ]
    n_mask  = random.randint(3, 5)
    to_mask = random.sample(angles, n_mask)

    for angle in to_mask:
        rad = np.radians(angle)
        x   = int(cx + r * np.cos(rad))
        y   = int(cy + r * np.sin(rad))
        size = 30
        draw.ellipse(
            [x-size, y-size, x+size, y+size],
            fill=255
        )

    return mask, n_mask

def visualize_mask(img_pil, mask_pil):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(img_pil)
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(mask_pil, cmap='gray')
    axes[1].set_title('Mask\n(white = generate new)')
    axes[1].axis('off')

    img_arr  = np.array(img_pil)
    mask_arr = np.array(mask_pil)
    overlay  = img_arr.copy()
    overlay[mask_arr > 128] = [255, 0, 0]
    axes[2].imshow(overlay)
    axes[2].set_title('Overlay\n(red = mask area)')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

sample_file = samples.iloc[0]['filename']
sample_img  = Image.open(
    f'{HUMAN_FOLDER}/{sample_file}'
).convert('RGB').resize((512, 512))

print('Hands Mask:')
hands_mask = create_hands_mask(sample_img)
visualize_mask(sample_img, hands_mask)

print('\nNumbers Mask:')
num_mask, n = create_numbers_mask(sample_img)
visualize_mask(sample_img, num_mask)
print(f'Masked {n} number positions')

In [ ]:
def create_numbers_mask_v2(img_pil, n_missing=4):
    w, h = img_pil.size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)

    cx = w // 2
    cy = h // 2
    r_clock = int(min(w, h) * 0.38)

    num_angles = {
        12: -90,  1: -60,  2: -30,
         3:   0,  4:  30,  5:  60,
         6:  90,  7: 120,  8: 150,
         9: 180, 10: 210, 11: 240,
    }

    numbers_to_mask = random.sample(
        list(num_angles.keys()), n_missing
    )

    size = int(min(w, h) * 0.08)

    masked_nums = []
    for num in numbers_to_mask:
        angle = num_angles[num]
        rad   = np.radians(angle)
        x = int(cx + r_clock * 0.85 * np.cos(rad))
        y = int(cy + r_clock * 0.85 * np.sin(rad))

        draw.ellipse(
            [x-size, y-size, x+size, y+size],
            fill=255
        )
        masked_nums.append(num)

    return mask, masked_nums

print('Numbers Mask v2:')
num_mask_v2, masked_nums = create_numbers_mask_v2(
    sample_img, n_missing=4
)
print(f'Masked numbers: {masked_nums}')
visualize_mask(sample_img, num_mask_v2)

In [ ]:
import cv2
import numpy as np

def find_clock_center(img_pil):
    """
    ใช้ OpenCV หา center และ radius
    """
    img_cv  = cv2.cvtColor(
        np.array(img_pil), cv2.COLOR_RGB2GRAY
    )
    blurred = cv2.GaussianBlur(img_cv, (9,9), 2)

    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1, minDist=50,
        param1=50, param2=30,
        minRadius=50, maxRadius=300
    )

    w, h = img_pil.size
    if circles is not None:
        circles = np.round(
            circles[0, :]
        ).astype(int)
        cx, cy, r = sorted(
            circles, key=lambda x: x[2]
        )[-1]
        return cx, cy, r
    else:
        return w//2, h//2, min(w,h)//3


def create_numbers_mask_v3(img_pil,
                            n_missing=4):
    w, h = img_pil.size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)

    cx, cy, r_clock = find_clock_center(img_pil)
    print(f'  Clock center: ({cx},{cy}) r={r_clock}')

    num_angles = {
        12: -90,  1: -60,  2: -30,
         3:   0,  4:  30,  5:  60,
         6:  90,  7: 120,  8: 150,
         9: 180, 10: 210, 11: 240,
    }

    numbers_to_mask = random.sample(
        list(num_angles.keys()), n_missing
    )

    size = max(15, int(r_clock * 0.18))

    for num in numbers_to_mask:
        angle = num_angles[num]
        rad   = np.radians(angle)
        x = int(cx + r_clock * 0.85 * np.cos(rad))
        y = int(cy + r_clock * 0.85 * np.sin(rad))

        x = max(size, min(w-size, x))
        y = max(size, min(h-size, y))

        draw.ellipse(
            [x-size, y-size, x+size, y+size],
            fill=255
        )

    return mask, numbers_to_mask

print('Numbers Mask v3:')
num_mask_v3, masked = create_numbers_mask_v3(
    sample_img, n_missing=4
)
print(f'Masked: {masked}')
visualize_mask(sample_img, num_mask_v3)

In [ ]:
def find_clock_center_v2(img_pil):
    """
    ใช้ contour หา bounding box
    ของวงนาฬิกา แม่นกว่า HoughCircles
    """
    img_cv = cv2.cvtColor(
        np.array(img_pil), cv2.COLOR_RGB2GRAY
    )
    w, h = img_pil.size

    _, thresh = cv2.threshold(
        img_cv, 200, 255, cv2.THRESH_BINARY_INV
    )

    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return w//2, h//2, min(w,h)//3

    largest = max(contours, key=cv2.contourArea)
    x, y, bw, bh = cv2.boundingRect(largest)

    cx = x + bw//2
    cy = y + bh//2
    r  = max(bw, bh)//2

    return cx, cy, r

cx, cy, r = find_clock_center_v2(sample_img)
print(f'Clock center: ({cx},{cy}) r={r}')

import matplotlib.patches as patches
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(sample_img)
circle = plt.Circle(
    (cx, cy), r,
    color='red', fill=False, linewidth=2
)
ax.add_patch(circle)
ax.plot(cx, cy, 'r+', markersize=15)
ax.set_title(f'Detected: center=({cx},{cy}) r={r}')
plt.show()

In [ ]:
def create_numbers_mask_v4(img_pil, n_missing=4):
    w, h = img_pil.size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)

    cx, cy, r_clock = find_clock_center_v2(img_pil)
    print(f'  Clock: ({cx},{cy}) r={r_clock}')

    num_angles = {
        12: -90,  1: -60,  2: -30,
         3:   0,  4:  30,  5:  60,
         6:  90,  7: 120,  8: 150,
         9: 180, 10: 210, 11: 240,
    }

    numbers_to_mask = random.sample(
        list(num_angles.keys()), n_missing
    )

    size = max(15, int(r_clock * 0.15))

    for num in numbers_to_mask:
        rad = np.radians(num_angles[num])
        x   = int(cx + r_clock * 0.72 * np.cos(rad))
        y   = int(cy + r_clock * 0.72 * np.sin(rad))
        x   = max(size, min(w-size, x))
        y   = max(size, min(h-size, y))
        draw.ellipse(
            [x-size, y-size, x+size, y+size],
            fill=255
        )

    return mask, numbers_to_mask


def create_hands_mask_v2(img_pil):
    w, h = img_pil.size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)

    cx, cy, r_clock = find_clock_center_v2(img_pil)
    r_mask = int(r_clock * 0.4)
    draw.ellipse(
        [cx-r_mask, cy-r_mask,
         cx+r_mask, cy+r_mask],
        fill=255
    )
    return mask

print('Hands Mask v2:')
hands_mask_v2 = create_hands_mask_v2(sample_img)
visualize_mask(sample_img, hands_mask_v2)

print('\nNumbers Mask v4:')
num_mask_v4, masked = create_numbers_mask_v4(
    sample_img, n_missing=4
)
print(f'Masked: {masked}')
visualize_mask(sample_img, num_mask_v4)

In [ ]:
def generate_class0_hands(img_pil, pipe,
                           n_samples=3):
    img_512  = img_pil.resize((512, 512))
    mask     = create_hands_mask(img_512)

    prompt = (
        "clock face without hands, "
        "hand drawn clock, blank center, "
        "no clock hands, pencil drawing, "
        "white paper background"
    )
    neg_prompt = (
        "clock hands, arrows, lines from center, "
        "colorful, digital"
    )

    results = []
    for _ in range(n_samples):
        out = pipe(
            prompt          = prompt,
            negative_prompt = neg_prompt,
            image           = img_512,
            mask_image      = mask,
            num_inference_steps=30,
            guidance_scale  = 7.5,
            strength        = 0.8,
        )
        results.append(out.images[0])

    return results, mask

def generate_class1_hands(img_pil, pipe,
                           n_samples=3):
    img_512  = img_pil.resize((512, 512))
    mask     = create_hands_mask(img_512)

    prompt = (
        "clock face with only one hand, "
        "hand drawn clock, single clock hand, "
        "pencil drawing, incomplete clock, "
        "white paper background"
    )
    neg_prompt = (
        "two hands, multiple hands, colorful, digital"
    )

    results = []
    for _ in range(n_samples):
        out = pipe(
            prompt          = prompt,
            negative_prompt = neg_prompt,
            image           = img_512,
            mask_image      = mask,
            num_inference_steps=30,
            guidance_scale  = 7.5,
            strength        = 0.8,
        )
        results.append(out.images[0])

    return results, mask

def generate_class0_digits(img_pil, pipe,
                            n_samples=3):
    img_512          = img_pil.resize((512, 512))
    mask, n_masked   = create_numbers_mask(img_512)

    prompt = (
        "clock face with missing numbers, "
        "hand drawn clock, incomplete numbers, "
        "blank spaces where numbers should be, "
        "pencil drawing, white paper"
    )
    neg_prompt = (
        "complete numbers, all 12 numbers, "
        "colorful, digital, printed"
    )

    results = []
    for _ in range(n_samples):
        out = pipe(
            prompt          = prompt,
            negative_prompt = neg_prompt,
            image           = img_512,
            mask_image      = mask,
            num_inference_steps=30,
            guidance_scale  = 7.5,
            strength        = 0.75,
        )
        results.append(out.images[0])

    return results, mask, n_masked

In [ ]:
test_imgs = perfect_df.sample(2, random_state=42)

all_generated = []

for _, row in test_imgs.iterrows():
    src     = f'{HUMAN_FOLDER}/{row["filename"]}'
    img_pil = Image.open(src).convert('RGB')

    print(f'\nProcessing: {row["filename"]}')

    print('  Generating class 0 hands...')
    gen_c0_hands, mask_h = generate_class0_hands(
        img_pil, pipe, n_samples=2
    )

    print('  Generating class 1 hands...')
    gen_c1_hands, _ = generate_class1_hands(
        img_pil, pipe, n_samples=2
    )

    print('  Generating class 0 digits...')
    gen_c0_digits, mask_d, n_m = \
        generate_class0_digits(
            img_pil, pipe, n_samples=2
        )

    all_generated.append({
        'filename'    : row['filename'],
        'original'    : img_pil,
        'c0_hands'    : gen_c0_hands,
        'c1_hands'    : gen_c1_hands,
        'c0_digits'   : gen_c0_digits,
        'mask_hands'  : mask_h,
        'mask_digits' : mask_d,
        'n_masked'    : n_m,
    })

In [ ]:
for data in all_generated:
    fname = data['filename']

    fig, axes = plt.subplots(
        3, 5, figsize=(25, 15)
    )

    # แถว 1: Original + Class 0 hands
    axes[0][0].imshow(data['original'])
    axes[0][0].set_title(
        f'Original\n{fname[:15]}', fontsize=9
    )
    axes[0][0].axis('off')

    axes[0][1].imshow(data['mask_hands'],
                      cmap='gray')
    axes[0][1].set_title('Mask (Hands)', fontsize=9)
    axes[0][1].axis('off')

    for i, img in enumerate(data['c0_hands'][:3]):
        axes[0][2+i].imshow(img)
        axes[0][2+i].set_title(
            f'Class 0 Hands\n(no hands) #{i+1}',
            fontsize=9
        )
        axes[0][2+i].axis('off')

    # แถว 2: Class 1 hands
    axes[1][0].imshow(data['original'])
    axes[1][0].set_title('Original', fontsize=9)
    axes[1][0].axis('off')

    axes[1][1].imshow(data['mask_hands'],
                      cmap='gray')
    axes[1][1].set_title('Mask (Hands)', fontsize=9)
    axes[1][1].axis('off')

    for i, img in enumerate(data['c1_hands'][:3]):
        axes[1][2+i].imshow(img)
        axes[1][2+i].set_title(
            f'Class 1 Hands\n(1 hand) #{i+1}',
            fontsize=9
        )
        axes[1][2+i].axis('off')

    # แถว 3: Class 0 digits
    axes[2][0].imshow(data['original'])
    axes[2][0].set_title('Original', fontsize=9)
    axes[2][0].axis('off')

    axes[2][1].imshow(data['mask_digits'],
                      cmap='gray')
    axes[2][1].set_title(
        f'Mask ({data["n_masked"]} digits)',
        fontsize=9
    )
    axes[2][1].axis('off')

    for i, img in enumerate(data['c0_digits'][:3]):
        axes[2][2+i].imshow(img)
        axes[2][2+i].set_title(
            f'Class 0 Digits\n(missing) #{i+1}',
            fontsize=9
        )
        axes[2][2+i].axis('off')

    plt.suptitle(
        f'Feasibility Test: {fname}\n'
        f'Row1=Class0 Hands | '
        f'Row2=Class1 Hands | '
        f'Row3=Class0 Digits',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        f'{RESULT_FOLDER}/gen_test_{fname[:10]}.png',
        dpi=120, bbox_inches='tight'
    )
    plt.show()

In [ ]:
# รัน cell นี้เฉพาะถ้าผล feasibility ดี

def save_generated(gen_imgs, base_filename,
                   class_type, target_folder):
    """บันทึกรูปที่ generate แล้ว"""
    saved = []
    for i, img in enumerate(gen_imgs):
        new_name = (
            f'gen_{class_type}_{i:03d}_'
            f'{base_filename}'
        )
        save_path = f'{target_folder}/{new_name}'
        img.resize((224, 224)).save(
            save_path, 'JPEG', quality=95
        )
        saved.append(new_name)
    return saved

# ถ้าผล feasibility ดี → generate เพิ่ม
# Target: เพิ่ม class 0 และ 1 ให้ได้ ~500 รูป/class

TARGET_PER_CLASS = 500
base_imgs = perfect_df.sample(
    min(100, len(perfect_df)), random_state=42
)

gen_log = {
    'c0_hands' : [],
    'c1_hands' : [],
    'c0_digits': [],
}

print(f'Generate target: {TARGET_PER_CLASS} รูป/class')
print('รัน cell นี้เฉพาะถ้า feasibility test ผ่าน!')